# Column with node-to-surface constraints — v6

Same column, solver, and load progression as v5. **New in v6**: the viewer now shows the **constraint infrastructure** alongside the deformed solid — phantom nodes, rigid-link connections, and the grounded `zeroLength` spring — all animated together across the load-history slider.

## Why?

`node_to_surface('base', start_face)` silently creates a hidden constraint topology under the hood:

- a **phantom node** co-located with every solid node on `start_face`,
- a **`rigidLink`** connecting the base reference point to each phantom,
- an **`equalDOF`** welding each phantom to its solid slave.

In v5 the viewer only sees the tet mesh and the 646 "public" nodes; the 44 phantom nodes, the rigid links, and the zeroLength base spring are invisible. That's fine for displacement post-processing, but you can't *see* what the constraint is actually doing to the geometry.

v6 fixes that by building an **augmented mesh** that adds:

| visual | cell kind | description |
|---|---|---|
| `vertex1` markers | phantom | one per phantom node |
| dashed lines | rigid_dash | N discrete `line2` segments per `rigidLink`, drawn from the ref point toward the phantom — linearly interpolated between the two end displacements |
| short stub line | zero_length | marks the `zeroLength` base spring |
| `vertex1` marker | ground | the fixed ground node |
| tet4 | tet | the original solid mesh, untouched |

The whole augmented grid is fed to `Results(...)` as a single time-series so the viewer slider animates every layer in sync. `equalDOF` pairs are *not* drawn — they are zero-length geometric degeneracies (phantom and slave coincide) and the phantom vertex markers already represent them.

## Knobs

```python
CONSTRAINT_SCALE = 40.0   # mm — offset for the ground marker below the base
N_DASHES         = 5      # dashes per rigid link
```

These are set in the augmented-mesh cell. Tweak `N_DASHES` for finer or coarser dashing, and `CONSTRAINT_SCALE` to push the ground marker further from the column base.

In [ ]:
from apeGmsh import apeGmsh, Part, Results
from apeGmsh import FEMData
from apeGmsh.sections import W_solid
import numpy as np
from pathlib import Path

In [ ]:
m1 = apeGmsh(model_name='beam_column', verbose=False)
m1.begin()

column = m1.sections.W_solid(bf=150, tf=20, h=300, tw=10, length=2000, label='column')

m1.model.selection.select_volumes().to_physical(name='pg_column')

base = m1.model.geometry.add_point(x=0, y=0, z=0,    lc=100, label='base')
top  = m1.model.geometry.add_point(x=0, y=0, z=2000, lc=100, label='top')

m1.constraints.node_to_surface('base', column.labels.start_face)
m1.constraints.node_to_surface('top',  column.labels.end_face)

m1.loads.point(
    target='top',
    force_xyz=[0, 1000, 0],
    moment_xyz=[0, 0, 0],
)

m1.mesh.sizing.set_global_size(100)
m1.mesh.generation.generate(dim=3)

fem = m1.mesh.queries.get_fem_data(remove_orphans=True)

m1.end()

## OpenSees model (same as v5)

- `ndf = 6` global, solid column nodes overridden to `ndf = 3`.
- Grounded helper node at `GROUND_TAG` fixed in 6 DOFs, connected to the base reference point via a stiff 6-DOF `zeroLength` spring.
- `constraints('Penalty')` because the phantom nodes are daisy-chained.

In [ ]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

ops.timeSeries('Linear', 1)

# --- Nodes -----------------------------------------------------------------
for nid, xyz in fem.nodes.get(target='pg_column'):
    ops.node(nid, *xyz, '-ndf', 3)

ref = fem.nodes.get(target=['base', 'top'])
for nid, xyz in ref:
    ops.node(nid, *xyz)

for nid, xyz in fem.nodes.constraints.phantom_nodes():
    ops.node(nid, *xyz)

# --- Grounded helper node + zeroLength spring at the base ------------------
base_id = int(next(iter(fem.nodes.get_ids(target='base'))))
base_xyz = next(xyz for nid, xyz in ref if nid == base_id)

GROUND_TAG = 10_000
ops.node(GROUND_TAG, *base_xyz)
ops.fix(GROUND_TAG, 1, 1, 1, 1, 1, 1)

K_SPRING = 1e14
for i in range(6):
    ops.uniaxialMaterial('Elastic', 100 + i, K_SPRING)

ZL_TAG = 20_000
ops.element('zeroLength', ZL_TAG, GROUND_TAG, base_id,
            '-mat', 100, 101, 102, 103, 104, 105,
            '-dir', 1, 2, 3, 4, 5, 6)

# --- Material + tet elements -----------------------------------------------
E  = 200e3
nu = 0.3
ops.nDMaterial('ElasticIsotropic', 1, E, nu)

for group in fem.elements.get(element_type='tet4'):
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', int(eid), *[int(n) for n in conn], 1)

# --- MP constraints --------------------------------------------------------
for master, slaves in fem.nodes.constraints.rigid_link_groups():
    for slave in slaves:
        ops.rigidLink('beam', int(master), int(slave))

for pair in fem.nodes.constraints.equal_dofs():
    ops.equalDOF(int(pair.master_node), int(pair.slave_node), *pair.dofs)

# --- Loads -----------------------------------------------------------------
ops.pattern('Plain', 1, 1)
for load in fem.nodes.loads.by_kind('nodal'):
    fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
    mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
    ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)

# --- Analysis --------------------------------------------------------------
ops.constraints('Penalty', 1e15, 1e15)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-6, 10)
ops.algorithm('Linear')
ops.integrator('LoadControl', 0.1)
ops.analysis('Static')

## Build the augmented points + cells array

The augmented grid keeps the original 646 solid nodes as index `0 .. n_solid-1`, appends the 44 phantom nodes (`n_solid .. n_solid + n_phantom - 1`), then the ground node, then any **extra points** used to draw dashed segments (two endpoints per dash).

Each extra point stores a triple `(src_a, src_b, t)` so its displacement can be reconstructed per step as a linear interpolation between `u(src_a)` and `u(src_b)`. For small displacements, linear blending along a rigid link is a faithful approximation of rigid-body motion and keeps the dashes moving correctly with the deformed shape.

Cells added:

- **tet** — one `VTK_TETRA` per tet4 in the solid mesh (unchanged).
- **rigid_dash** — each `rigidLink` contributes `N_DASHES` `VTK_LINE` cells, whose `2 * N_DASHES` endpoints become new extra points anchored to the (master, slave) pair.
- **phantom** — one `VTK_VERTEX` per phantom node.
- **zero_length** — one `VTK_LINE` between the ground index and the base-reference index.
- **ground** — one `VTK_VERTEX` at the ground index.

A `constraint_kind` cell field tags every cell with its kind so the viewer can colour-separate them via the **Contour** dropdown.

In [ ]:
CONSTRAINT_SCALE = 40.0   # mm — ground-marker offset below base
N_DASHES         = 5      # dashes per rigid link

solid_ids    = [int(nid) for nid in fem.nodes.ids]
solid_coords = np.asarray(fem.nodes.coords, dtype=np.float64)
n_solid      = len(solid_ids)

_pn = fem.nodes.constraints.phantom_nodes()
ph_ids_raw, ph_coords_raw = _pn.ids, _pn.coords
phantom_ids    = [int(p) for p in ph_ids_raw]
phantom_coords = np.asarray(ph_coords_raw, dtype=np.float64)
n_phantom      = len(phantom_ids)

ground_xyz = np.array([base_xyz[0], base_xyz[1], base_xyz[2] - CONSTRAINT_SCALE * 2.0])

solid_id_to_idx   = {nid: i for i, nid in enumerate(solid_ids)}
phantom_id_to_idx = {nid: n_solid + i for i, nid in enumerate(phantom_ids)}
ground_idx        = n_solid + n_phantom

def node_idx(nid):
    """Resolve a node id (solid or phantom) to its augmented-grid index."""
    if nid in solid_id_to_idx:
        return solid_id_to_idx[nid]
    if nid in phantom_id_to_idx:
        return phantom_id_to_idx[nid]
    return None

def node_xyz(nid):
    if nid in solid_id_to_idx:
        return solid_coords[solid_id_to_idx[nid]]
    if nid in phantom_id_to_idx:
        return phantom_coords[phantom_id_to_idx[nid] - n_solid]
    return None

extra_points: list[np.ndarray]                = []
extra_point_src: list[tuple[int, int, float]] = []

def add_extra_point(xyz, src_a, src_b, t):
    gi = n_solid + n_phantom + 1 + len(extra_points)
    extra_points.append(np.asarray(xyz, dtype=np.float64))
    extra_point_src.append((int(src_a), int(src_b), float(t)))
    return gi

cells_list:     list[list[int]] = []
celltypes_list: list[int]       = []
kinds_list:     list[int]       = []

KIND_TET        = 0
KIND_RIGID_DASH = 1
KIND_PHANTOM    = 2
KIND_ZEROLENGTH = 3
KIND_GROUND     = 4

# ── Tets ─────────────────────────────────────────────────────────
for group in fem.elements:
    if group.dim != 3 or group.connectivity.size == 0:
        continue
    for row in group.connectivity:
        idxs = [solid_id_to_idx[int(n)] for n in row]
        cells_list.append([4] + idxs)
        celltypes_list.append(10)   # VTK_TETRA
        kinds_list.append(KIND_TET)

# ── Rigid links (dashed line segments between ref point and phantom)
for master_nid, slaves in fem.nodes.constraints.rigid_link_groups():
    m_nid = int(master_nid)
    p0 = node_xyz(m_nid)
    if p0 is None:
        continue
    for slave_nid in slaves:
        s_nid = int(slave_nid)
        p1 = node_xyz(s_nid)
        if p1 is None:
            continue
        L = float(np.linalg.norm(p1 - p0))
        if L < 1e-9:
            continue
        direction = (p1 - p0) / L

        # N dashes + (N-1) equal gaps
        total_segments = 2 * N_DASHES - 1
        seg_len = L / total_segments

        for k in range(N_DASHES):
            d0 = (2 * k) * seg_len
            d1 = (2 * k + 1) * seg_len
            t0 = d0 / L
            t1 = d1 / L
            start_xyz = p0 + direction * d0
            end_xyz   = p0 + direction * d1
            i0 = add_extra_point(start_xyz, m_nid, s_nid, t0)
            i1 = add_extra_point(end_xyz,   m_nid, s_nid, t1)
            cells_list.append([2, i0, i1])
            celltypes_list.append(3)    # VTK_LINE
            kinds_list.append(KIND_RIGID_DASH)

# ── Phantom markers (vertex1) ───────────────────────────────────
for ph_nid in phantom_ids:
    cells_list.append([1, phantom_id_to_idx[ph_nid]])
    celltypes_list.append(1)    # VTK_VERTEX
    kinds_list.append(KIND_PHANTOM)

# ── ZeroLength stub + ground marker ─────────────────────────────
cells_list.append([2, ground_idx, solid_id_to_idx[base_id]])
celltypes_list.append(3)
kinds_list.append(KIND_ZEROLENGTH)

cells_list.append([1, ground_idx])
celltypes_list.append(1)
kinds_list.append(KIND_GROUND)

# ── Assemble arrays ─────────────────────────────────────────────
n_extras = len(extra_points)
all_points = np.vstack([
    solid_coords,
    phantom_coords,
    ground_xyz.reshape(1, 3),
    np.asarray(extra_points) if n_extras else np.zeros((0, 3)),
])
n_total_points = all_points.shape[0]

cells_flat = np.concatenate([np.asarray(c, dtype=np.int64) for c in cells_list])
cell_types = np.asarray(celltypes_list, dtype=np.uint8)
constraint_kind = np.asarray(kinds_list, dtype=np.int32)

print(f'Augmented grid: {n_total_points} pts, {len(cell_types)} cells')
print(f'  solid nodes:   {n_solid}')
print(f'  phantom nodes: {n_phantom}')
print(f'  ground:        1')
print(f'  extras (dash): {n_extras}')
print(f'\nCell kinds:')
for k, name in enumerate(['tet','rigid_dash','phantom','zero_length','ground']):
    c = int(np.sum(constraint_kind == k))
    print(f'  {k} {name:<12}: {c}')

## Stepwise analysis — capture solid + phantom displacements

Same 10-step `LoadControl` loop as v5, but per step we now collect:

1. Displacement for each solid node (3-DOF query → `u_all[0 : n_solid]`).
2. Displacement for each phantom node (6-DOF query, translation components only → `u_all[n_solid : n_solid + n_phantom]`).
3. The ground index stays at `(0, 0, 0)` — it's fixed.
4. Every extra point uses its `(src_a, src_b, t)` triple to blend the displacements of its source nodes linearly. This is the glue that makes the dashes track the rigid links through the loading history.

In [ ]:
n_steps = 10
time_values: list[float]        = []
step_disp_list: list[np.ndarray] = []

for i in range(n_steps):
    ok = ops.analyze(1)
    if ok != 0:
        print(f'Step {i+1}: failed ({ok})')
        break

    t = float(ops.getTime())
    time_values.append(t)

    u_all = np.zeros((n_total_points, 3), dtype=np.float64)

    # Solid nodes
    for j, nid in enumerate(solid_ids):
        d = ops.nodeDisp(nid)
        u_all[j, 0] = d[0]
        u_all[j, 1] = d[1]
        u_all[j, 2] = d[2]

    # Phantom nodes (6-DOF — take translation components)
    for k_idx, ph_nid in enumerate(phantom_ids):
        d = ops.nodeDisp(ph_nid)
        u_all[n_solid + k_idx, 0] = d[0]
        u_all[n_solid + k_idx, 1] = d[1]
        u_all[n_solid + k_idx, 2] = d[2]

    # Ground stays at zero (pre-initialised).

    # Extra dash-endpoint points: linear interpolation src_a -> src_b
    for ex_i, (src_a, src_b, tt) in enumerate(extra_point_src):
        gi = n_solid + n_phantom + 1 + ex_i
        idx_a = node_idx(src_a)
        idx_b = node_idx(src_b)
        if idx_a is None or idx_b is None:
            continue
        u_all[gi] = (1.0 - tt) * u_all[idx_a] + tt * u_all[idx_b]

    step_disp_list.append(u_all)
    print(f'Step {i+1:2d}  λ = {t:.2f}   |U|_max = {np.linalg.norm(u_all, axis=1).max():.4e}')

print(f'\nCaptured {len(step_disp_list)} steps.')

## Base reaction at the final step

Expected from equilibrium: `Fy = -1000 N`, `Mx = +2·10⁶ N·mm`.

In [ ]:
ops.reactions()

rxn = ops.nodeReaction(GROUND_TAG)

print(f'nodeReaction(ground = {GROUND_TAG}):')
print(f'  Fx = {rxn[0]:+.4e}   Fy = {rxn[1]:+.4e}   Fz = {rxn[2]:+.4e}')
print(f'  Mx = {rxn[3]:+.4e}   My = {rxn[4]:+.4e}   Mz = {rxn[5]:+.4e}')
print(f'\nExpected:  Fy = -1.0000e+03   Mx = +2.0000e+06')

## Wrap into a time-series Results and launch the viewer

The augmented grid can't be built through `Results.from_fem(fem, steps=...)` — that path derives connectivity from `fem.elements` and doesn't know about phantoms, dashes, or the ground stub. Instead we call `Results(...)` directly with the pre-built `cells`, `cell_types`, and step dicts.

Per-step point fields (`Displacement`, `|U|`, `Ux`, `Uy`, `Uz`) are computed from `step_disp_list`. The `constraint_kind` cell field is constant across steps, so we repeat it `n_steps` times to match the step-count.

In [ ]:
step_point_fields = {
    'Displacement': step_disp_list,
    '|U|':          [np.linalg.norm(u, axis=1) for u in step_disp_list],
    'Ux':           [u[:, 0].copy() for u in step_disp_list],
    'Uy':           [u[:, 1].copy() for u in step_disp_list],
    'Uz':           [u[:, 2].copy() for u in step_disp_list],
}

step_cell_fields = {
    'constraint_kind': [constraint_kind.astype(np.float64)] * len(time_values),
}

results = Results(
    node_coords=all_points,
    cells=cells_flat,
    cell_types=cell_types,
    time_steps=time_values,
    step_point_fields=step_point_fields,
    step_cell_fields=step_cell_fields,
    name='column_nts_v6',
)

print(f'Results: {len(time_values)} steps, {n_total_points} nodes, {len(cell_types)} cells')
print(f'Time range: [{time_values[0]:.2f}, {time_values[-1]:.2f}]')
results

In [ ]:
results.viewer(blocking=False)

## Spot-check the augmented field

Quick sanity: the phantom nodes should develop non-zero displacement (they follow the top reference point), and the first dashed-line endpoint should match its `(src_a, src_b, t)` blend exactly.

In [ ]:
last_u = step_disp_list[-1]

top_id   = int(next(iter(fem.nodes.get_ids(target='top'))))
top_idx  = solid_id_to_idx[top_id]
print(f'Top node (id={top_id}, idx={top_idx}) final disp:')
print(f'  {last_u[top_idx]}')

ph_start = n_solid
ph_end   = n_solid + n_phantom
ph_umag  = np.linalg.norm(last_u[ph_start:ph_end], axis=1)
print(f'\nPhantom |U| range (final step): [{ph_umag.min():.4e}, {ph_umag.max():.4e}]')

if n_extras:
    ex_start = n_solid + n_phantom + 1
    src_a, src_b, tt = extra_point_src[0]
    idx_a = node_idx(src_a)
    idx_b = node_idx(src_b)
    expected = (1.0 - tt) * last_u[idx_a] + tt * last_u[idx_b]
    print(f'\nFirst extra point blend check:')
    print(f'  src_a={src_a}, src_b={src_b}, t={tt:.3f}')
    print(f'  expected = {expected}')
    print(f'  actual   = {last_u[ex_start]}')
    print(f'  match    = {np.allclose(expected, last_u[ex_start])}')

## Linearity check

The system is linear, so `|U|_max` should scale exactly proportional to the load factor λ. Residuals should be ~0.

In [ ]:
lambdas = np.array(time_values)
umax    = np.array([np.linalg.norm(u, axis=1).max() for u in step_disp_list])

print('  step    λ        |U|_max')
print('  ----  ------  -----------')
for i, (lam, u) in enumerate(zip(lambdas, umax), 1):
    print(f'  {i:>4}  {lam:6.2f}  {u:11.4e}')

ratio = umax[-1] / lambdas[-1] if lambdas[-1] else 0
residuals = np.abs(umax - ratio * lambdas)
print(f'\n|U|_max / λ (const):   {ratio:.4e}')
print(f'max abs residual:      {residuals.max():.3e}')

## 9. Capture results — manual + DomainCapture paths

Two ways to produce a native-HDF5 results file consumable by the
apeGmsh ``ResultsViewer``:

1. **Manual path** — query the live OpenSees domain post-analysis,
   open a ``NativeWriter``, and write nodal displacements yourself.
   Good for one-shot snapshots and post-hoc diagnostics.
2. **DomainCapture path** — declare what to capture with
   ``Recorders().nodes(...)``, hand the spec to a ``DomainCapture``
   context, and call ``cap.step(t=...)`` after each ``ops.analyze``
   (the helper does it for you). Scales to multi-stage, transient,
   modal, and multi-recorder runs.

Both produce a file that ``Results.from_native(path).viewer()`` can
open. The viewer launch is gated on ``APEGMSH_SKIP_VIEWER`` so this
notebook is safe to run under nbconvert / CI.


In [ ]:
# --- EOS-WIRING-V1 ---
# Manual path: pull displacements off the live domain, write h5 yourself.
from pathlib import Path
import numpy as np
from apeGmsh.results.writers import NativeWriter

results_manual = Path("example_column_nodeToSurface_v6_manual.h5")
if results_manual.exists():
    results_manual.unlink()

_n = len(fem.nodes.ids)
_ux = np.array([ops.nodeDisp(int(nid), 1) for nid in fem.nodes.ids])
_uy = np.array([ops.nodeDisp(int(nid), 2) for nid in fem.nodes.ids])
_uz = np.array([ops.nodeDisp(int(nid), 3) for nid in fem.nodes.ids])
_components = {
    "displacement_x": _ux.reshape(1, _n),
    "displacement_y": _uy.reshape(1, _n),
    "displacement_z": _uz.reshape(1, _n),
}

with NativeWriter(results_manual) as _nw:
    _nw.open(fem=fem)
    _sid = _nw.begin_stage(name="static", kind="static", time=np.array([1.0]))
    _nw.write_nodes(
        _sid, "partition_0",
        node_ids=np.asarray(fem.nodes.ids, dtype=np.int64),
        components=_components,
    )
    _nw.end_stage()

print(f"manual -> {results_manual} ({results_manual.stat().st_size/1024:.1f} KB)")


In [ ]:
# DomainCapture path: declarative recorders, capture during analyze.
from apeGmsh.solvers.Recorders import Recorders
from apeGmsh.results.capture._domain import DomainCapture

recs = Recorders()
recs.nodes(components="displacement")
recs.nodes(components="reaction_force")
spec = recs.resolve(fem, ndm=3, ndf=6)

results_capture = Path("example_column_nodeToSurface_v6_capture.h5")
if results_capture.exists():
    results_capture.unlink()

with DomainCapture(spec, results_capture, fem, ndm=3, ndf=6) as cap:
    cap.begin_stage("run", kind="static")
    # --- copied from cell 4 ---
    import openseespy.opensees as ops

    ops.wipe()
    ops.model('basic', '-ndm', 3, '-ndf', 6)

    ops.timeSeries('Linear', 1)

    # --- Nodes -----------------------------------------------------------------
    for nid, xyz in fem.nodes.get(target='pg_column'):
        ops.node(nid, *xyz, '-ndf', 3)

    ref = fem.nodes.get(target=['base', 'top'])
    for nid, xyz in ref:
        ops.node(nid, *xyz)

    for nid, xyz in fem.nodes.constraints.phantom_nodes():
        ops.node(nid, *xyz)

    # --- Grounded helper node + zeroLength spring at the base ------------------
    base_id = int(next(iter(fem.nodes.get_ids(target='base'))))
    base_xyz = next(xyz for nid, xyz in ref if nid == base_id)

    GROUND_TAG = 10_000
    ops.node(GROUND_TAG, *base_xyz)
    ops.fix(GROUND_TAG, 1, 1, 1, 1, 1, 1)

    K_SPRING = 1e14
    for i in range(6):
        ops.uniaxialMaterial('Elastic', 100 + i, K_SPRING)

    ZL_TAG = 20_000
    ops.element('zeroLength', ZL_TAG, GROUND_TAG, base_id,
                '-mat', 100, 101, 102, 103, 104, 105,
                '-dir', 1, 2, 3, 4, 5, 6)

    # --- Material + tet elements -----------------------------------------------
    E  = 200e3
    nu = 0.3
    ops.nDMaterial('ElasticIsotropic', 1, E, nu)

    for group in fem.elements.get(element_type='tet4'):
        for eid, conn in group:
            ops.element('FourNodeTetrahedron', int(eid), *[int(n) for n in conn], 1)

    # --- MP constraints --------------------------------------------------------
    for master, slaves in fem.nodes.constraints.rigid_link_groups():
        for slave in slaves:
            ops.rigidLink('beam', int(master), int(slave))

    for pair in fem.nodes.constraints.equal_dofs():
        ops.equalDOF(int(pair.master_node), int(pair.slave_node), *pair.dofs)

    # --- Loads -----------------------------------------------------------------
    ops.pattern('Plain', 1, 1)
    for load in fem.nodes.loads.by_kind('nodal'):
        fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
        mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
        ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)

    # --- Analysis --------------------------------------------------------------
    ops.constraints('Penalty', 1e15, 1e15)
    ops.numberer('RCM')
    ops.system('UmfPack')
    ops.test('NormDispIncr', 1.0e-6, 10)
    ops.algorithm('Linear')
    ops.integrator('LoadControl', 0.1)
    ops.analysis('Static')
    # --- copied from cell 6 ---
    CONSTRAINT_SCALE = 40.0   # mm — ground-marker offset below base
    N_DASHES         = 5      # dashes per rigid link

    solid_ids    = [int(nid) for nid in fem.nodes.ids]
    solid_coords = np.asarray(fem.nodes.coords, dtype=np.float64)
    n_solid      = len(solid_ids)

    _pn = fem.nodes.constraints.phantom_nodes()
    ph_ids_raw, ph_coords_raw = _pn.ids, _pn.coords
    phantom_ids    = [int(p) for p in ph_ids_raw]
    phantom_coords = np.asarray(ph_coords_raw, dtype=np.float64)
    n_phantom      = len(phantom_ids)

    ground_xyz = np.array([base_xyz[0], base_xyz[1], base_xyz[2] - CONSTRAINT_SCALE * 2.0])

    solid_id_to_idx   = {nid: i for i, nid in enumerate(solid_ids)}
    phantom_id_to_idx = {nid: n_solid + i for i, nid in enumerate(phantom_ids)}
    ground_idx        = n_solid + n_phantom

    def node_idx(nid):
        """Resolve a node id (solid or phantom) to its augmented-grid index."""
        if nid in solid_id_to_idx:
            return solid_id_to_idx[nid]
        if nid in phantom_id_to_idx:
            return phantom_id_to_idx[nid]
        return None

    def node_xyz(nid):
        if nid in solid_id_to_idx:
            return solid_coords[solid_id_to_idx[nid]]
        if nid in phantom_id_to_idx:
            return phantom_coords[phantom_id_to_idx[nid] - n_solid]
        return None

    extra_points: list[np.ndarray]                = []
    extra_point_src: list[tuple[int, int, float]] = []

    def add_extra_point(xyz, src_a, src_b, t):
        gi = n_solid + n_phantom + 1 + len(extra_points)
        extra_points.append(np.asarray(xyz, dtype=np.float64))
        extra_point_src.append((int(src_a), int(src_b), float(t)))
        return gi

    cells_list:     list[list[int]] = []
    celltypes_list: list[int]       = []
    kinds_list:     list[int]       = []

    KIND_TET        = 0
    KIND_RIGID_DASH = 1
    KIND_PHANTOM    = 2
    KIND_ZEROLENGTH = 3
    KIND_GROUND     = 4

    # ── Tets ─────────────────────────────────────────────────────────
    for group in fem.elements:
        if group.dim != 3 or group.connectivity.size == 0:
            continue
        for row in group.connectivity:
            idxs = [solid_id_to_idx[int(n)] for n in row]
            cells_list.append([4] + idxs)
            celltypes_list.append(10)   # VTK_TETRA
            kinds_list.append(KIND_TET)

    # ── Rigid links (dashed line segments between ref point and phantom)
    for master_nid, slaves in fem.nodes.constraints.rigid_link_groups():
        m_nid = int(master_nid)
        p0 = node_xyz(m_nid)
        if p0 is None:
            continue
        for slave_nid in slaves:
            s_nid = int(slave_nid)
            p1 = node_xyz(s_nid)
            if p1 is None:
                continue
            L = float(np.linalg.norm(p1 - p0))
            if L < 1e-9:
                continue
            direction = (p1 - p0) / L

            # N dashes + (N-1) equal gaps
            total_segments = 2 * N_DASHES - 1
            seg_len = L / total_segments

            for k in range(N_DASHES):
                d0 = (2 * k) * seg_len
                d1 = (2 * k + 1) * seg_len
                t0 = d0 / L
                t1 = d1 / L
                start_xyz = p0 + direction * d0
                end_xyz   = p0 + direction * d1
                i0 = add_extra_point(start_xyz, m_nid, s_nid, t0)
                i1 = add_extra_point(end_xyz,   m_nid, s_nid, t1)
                cells_list.append([2, i0, i1])
                celltypes_list.append(3)    # VTK_LINE
                kinds_list.append(KIND_RIGID_DASH)

    # ── Phantom markers (vertex1) ───────────────────────────────────
    for ph_nid in phantom_ids:
        cells_list.append([1, phantom_id_to_idx[ph_nid]])
        celltypes_list.append(1)    # VTK_VERTEX
        kinds_list.append(KIND_PHANTOM)

    # ── ZeroLength stub + ground marker ─────────────────────────────
    cells_list.append([2, ground_idx, solid_id_to_idx[base_id]])
    celltypes_list.append(3)
    kinds_list.append(KIND_ZEROLENGTH)

    cells_list.append([1, ground_idx])
    celltypes_list.append(1)
    kinds_list.append(KIND_GROUND)

    # ── Assemble arrays ─────────────────────────────────────────────
    n_extras = len(extra_points)
    all_points = np.vstack([
        solid_coords,
        phantom_coords,
        ground_xyz.reshape(1, 3),
        np.asarray(extra_points) if n_extras else np.zeros((0, 3)),
    ])
    n_total_points = all_points.shape[0]

    cells_flat = np.concatenate([np.asarray(c, dtype=np.int64) for c in cells_list])
    cell_types = np.asarray(celltypes_list, dtype=np.uint8)
    constraint_kind = np.asarray(kinds_list, dtype=np.int32)

    print(f'Augmented grid: {n_total_points} pts, {len(cell_types)} cells')
    print(f'  solid nodes:   {n_solid}')
    print(f'  phantom nodes: {n_phantom}')
    print(f'  ground:        1')
    print(f'  extras (dash): {n_extras}')
    print(f'\nCell kinds:')
    for k, name in enumerate(['tet','rigid_dash','phantom','zero_length','ground']):
        c = int(np.sum(constraint_kind == k))
        print(f'  {k} {name:<12}: {c}')
    # --- copied from cell 8 ---
    n_steps = 10
    time_values: list[float]        = []
    step_disp_list: list[np.ndarray] = []

    for i in range(n_steps):
        ok = ops.analyze(1)
        cap.step(t=ops.getTime())
        if ok != 0:
            print(f'Step {i+1}: failed ({ok})')
            break

        t = float(ops.getTime())
        time_values.append(t)

        u_all = np.zeros((n_total_points, 3), dtype=np.float64)

        # Solid nodes
        for j, nid in enumerate(solid_ids):
            d = ops.nodeDisp(nid)
            u_all[j, 0] = d[0]
            u_all[j, 1] = d[1]
            u_all[j, 2] = d[2]

        # Phantom nodes (6-DOF — take translation components)
        for k_idx, ph_nid in enumerate(phantom_ids):
            d = ops.nodeDisp(ph_nid)
            u_all[n_solid + k_idx, 0] = d[0]
            u_all[n_solid + k_idx, 1] = d[1]
            u_all[n_solid + k_idx, 2] = d[2]

        # Ground stays at zero (pre-initialised).

        # Extra dash-endpoint points: linear interpolation src_a -> src_b
        for ex_i, (src_a, src_b, tt) in enumerate(extra_point_src):
            gi = n_solid + n_phantom + 1 + ex_i
            idx_a = node_idx(src_a)
            idx_b = node_idx(src_b)
            if idx_a is None or idx_b is None:
                continue
            u_all[gi] = (1.0 - tt) * u_all[idx_a] + tt * u_all[idx_b]

        step_disp_list.append(u_all)
        print(f'Step {i+1:2d}  λ = {t:.2f}   |U|_max = {np.linalg.norm(u_all, axis=1).max():.4e}')

    print(f'\nCaptured {len(step_disp_list)} steps.')
    cap.end_stage()

print(f"capture -> {results_capture} ({results_capture.stat().st_size/1024:.1f} KB)")


In [ ]:
# Open the captured run in the apeGmsh ResultsViewer (subprocess,
# non-blocking). Set APEGMSH_SKIP_VIEWER=1 to skip in headless / CI.
import os
from apeGmsh.results import Results
results = Results.from_native(results_capture)
if os.environ.get("APEGMSH_SKIP_VIEWER"):
    print("[skip viewer] APEGMSH_SKIP_VIEWER set")
else:
    handle = results.viewer(blocking=False)
    print(f"viewer pid: {handle.pid}  -- close window to exit.")
